[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Deep-unlearning/smol-audio/blob/master/cohere_transcribe_streaming/Stream_Cohere_Transcribe.ipynb)

# Stream Cohere Transcribe, live

[Cohere Transcribe](https://huggingface.co/CohereLabs/cohere-transcribe-03-2026) is a 2B speech-to-text model that
needs the **whole recording** before it can answer. **Cohere Streaming**
([`Steveeeeeeen/cohere-streaming`](https://huggingface.co/Steveeeeeeen/cohere-streaming)) is a version of it that
listens **as the audio arrives** and writes captions while you are still speaking.

In this notebook you will:

1. install the dependencies and load the streaming model,
2. download a short speech clip,
3. feed it to the model in tiny 80 ms pieces and watch the transcript grow,
4. get the final, polished transcript,
5. compare different latency settings (`lookahead` = 0, 1, 3, 6, 13),
6. (optional) measure word error rate with `jiwer`.

**Hardware.** Use a GPU runtime (Colab: *Runtime > Change runtime type > T4 GPU* or better). The model has 2B
parameters, so on a CPU it works but is much slower than real time. Loading also temporarily holds the original
offline encoder and its streaming replacement together in host memory: allow generous system RAM as well as GPU
memory. The CPU path is useful for learning the API when sufficient RAM is available.

This example is part of [smol-audio](https://github.com/Deep-unlearning/smol-audio/tree/master/cohere_transcribe_streaming), adapted from [the upstream notebook contribution](https://github.com/Deep-unlearning/cohere-transcribe-streaming/tree/1e1b41ca259b85d0e2b257b6a794d1b9c1e1ef2d) ([PR #3](https://github.com/Deep-unlearning/cohere-transcribe-streaming/pull/3)).

## 1. Setup

The code lives in the `cohere_transcribe_streaming` folder of [smol-audio](https://github.com/Deep-unlearning/smol-audio).
The cell below clones the repo in a fresh Colab runtime, or reuses your local checkout, then moves into that folder
and installs the dependencies from `requirements.txt`.


In [ ]:
from pathlib import Path
import subprocess

# Reuse a local checkout, or clone smol-audio in a fresh Colab runtime. Safe to rerun this cell.
here = Path.cwd()
if here.name == "cohere_transcribe_streaming" and (here / "model.py").is_file():
    example_dir = here
elif (here / "cohere_transcribe_streaming" / "model.py").is_file():
    example_dir = here / "cohere_transcribe_streaming"
else:
    checkout = here / "smol-audio"
    if not checkout.exists():
        subprocess.run(["git", "clone", "--quiet", "https://github.com/Deep-unlearning/smol-audio", str(checkout)], check=True)
    example_dir = checkout / "cohere_transcribe_streaming"
%cd {example_dir}
%pip install -r requirements.txt


Before downloading the model, sign in on the [Cohere Transcribe model page](https://huggingface.co/CohereLabs/cohere-transcribe-03-2026)
and accept its access conditions. Use a [Hugging Face read token](https://huggingface.co/settings/tokens) from that same account.

The cell below opens the login form only if no token is already available. In Colab, you can instead add an `HF_TOKEN`
secret in the Secrets panel and enable notebook access; the Hub library will use it automatically.


In [ ]:
from huggingface_hub import get_token, notebook_login

if get_token() is None:
    notebook_login()


Let's check which device we have. We choose native bf16 on recent GPUs, fp16 on a T4, and float32 on a CPU.
A T4 can store bf16 tensors but does not have native bf16 compute support, so it uses fp16 here.

In [ ]:
import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
NATIVE_BF16 = DEVICE == "cuda" and torch.cuda.is_bf16_supported(including_emulation=False)
if NATIVE_BF16:
    DTYPE = torch.bfloat16
elif DEVICE == "cuda":
    DTYPE = torch.float16
else:
    DTYPE = torch.float32
FINAL_REENCODE_MAX_S = 30.0

print("torch", torch.__version__)
print("device:", DEVICE)
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
print("inference dtype:", DTYPE)

## 2. Get a speech clip

We use a clip from a tiny LibriSpeech test set (read English audiobooks) hosted on the Hub. We pick one short clip,
keep its reference text (what the speaker really said) and play it.

The model expects **16 kHz mono** audio as float numbers between -1 and 1. LibriSpeech already is 16 kHz.


In [ ]:
import io
import numpy as np
import soundfile as sf
import pyarrow.parquet as pq
from huggingface_hub import hf_hub_download
from IPython.display import Audio

SR = 16000  # the model works at 16 kHz

parquet = hf_hub_download(
    "hf-internal-testing/librispeech_asr_dummy",
    "clean/validation-00000-of-00001.parquet",
    repo_type="dataset",
)
rows = pq.read_table(parquet, columns=["audio", "text"]).to_pylist()

def decode(row):
    wav, sr = sf.read(io.BytesIO(row["audio"]["bytes"]), dtype="float32")
    assert sr == SR
    return wav

audio = decode(rows[1])
reference = rows[1]["text"]
print(f"{len(audio) / SR:.1f} seconds of audio")
print("REFERENCE:", reference)
Audio(audio, rate=SR)

## 3. Concepts in plain words

Before we run anything, three ideas.

**80 ms chunks.** During live streaming, audio arrives in small slices, and the model eats
one slice at a time. Here each slice is **80 ms** long, which is `1280` samples at 16 kHz (that is
12.5 slices per second). Think of it as the model's heartbeat: every 80 ms it takes in a bit of sound and updates its
internal memory (a cache). At the endpoint, short clips also get a fresh encoder pass for the final transcript.

**Lookahead.** Understanding a sound sometimes depends on what comes right after it (is that "ice cream" or "I scream"?).
`lookahead` is how many extra 80 ms slices of *future* audio the model waits for before it commits to its
interpretation of a moment. More lookahead means better accuracy but later captions. The encoder
waits for `(lookahead + 1) x 80 ms` of audio per update:

| lookahead | 0 | 1 | 3 | 6 | 13 |
|---|---|---|---|---|---|
| encoder audio wait | 80 ms | 160 ms | 320 ms | 560 ms | 1.12 s |

One model supports all of these. You choose at load time. These numbers describe the encoder audio wait;
caption delay also includes model compute, partial decoding, and word commitment. Actual timing depends on hardware.

**Live vs final transcript.** There are two kinds of output:

* **Live** text (`model.step(chunk)`): a running best guess, updated as audio comes in. The last word or two can still change.
* **Final** text (`model.finish()`): with the default AED partials and free final decoding, short utterances get
  a fresh encoder pass at `final_lookahead=13`, then beam search. This makes the final decode independent of the live
  `lookahead` while the buffered utterance fits `final_reencode_max_s` (30 s by default). For longer utterances, the
  model uses its streamed encoder states, so the final result can depend on the live lookahead.

The final pass takes additional time after the speaker stops. We measure it below on your hardware; its duration
also depends on clip length, transcript length and decoding settings.

## 4. Load the streaming model

`CohereStreaming(lookahead=6)` downloads two things from the Hub: the original Cohere Transcribe weights (the decoder and
tokenizer) and the streaming checkpoint (the cache-aware encoder). The first call takes a few minutes. A lookahead of 6
gives an encoder audio wait of 560 ms; visible caption timing also includes compute and decoding.

In [ ]:
from model import CohereStreaming

model = CohereStreaming(lookahead=6, device=DEVICE, dtype=DTYPE, final_reencode_max_s=FINAL_REENCODE_MAX_S)
print("device:", model.device, "| chunk size:", model.hop_ms, "ms")


## 5. Stream the clip, 80 ms at a time

The protocol has three calls:

* `model.reset(language="en")` starts a new utterance (languages: en, fr, de, es, it, pt, nl, pl),
* `model.step(chunk)` feeds one 80 ms chunk and returns the **current live transcript**,
* `model.finish()` signals the end of speech and returns the **final transcript**.

Below we cut the clip into 1280-sample chunks and loop. We only print when the text changed, together with the
position in the audio (not wall-clock time) where it changed. Watch how words appear and get corrected, for example a
partial word like "Quill" becoming "Quilter's".

In [ ]:
import time

def synchronize_cuda():
    """Wait for queued GPU work before measuring elapsed time."""
    if DEVICE == "cuda":
        torch.cuda.synchronize()

def stream(model, audio, language="en", verbose=False):
    """Return the caption changes, final text, and timing for one clip."""
    chunk_samples = SR * model.hop_ms // 1000
    model.reset(language=language)
    trace = []
    step_times = []
    previous_text = None
    for start in range(0, len(audio), chunk_samples):
        chunk = audio[start:start + chunk_samples]
        if len(chunk) < chunk_samples:
            chunk = np.pad(chunk, (0, chunk_samples - len(chunk)))
        synchronize_cuda()
        step_started = time.perf_counter()
        text = model.step(chunk)
        synchronize_cuda()
        step_times.append(time.perf_counter() - step_started)
        audio_time = (start + chunk_samples) / SR
        if text != previous_text:
            trace.append((audio_time, text))
            if verbose:
                print(f"[{audio_time:5.2f}s] {text}")
            previous_text = text
    synchronize_cuda()
    finish_started = time.perf_counter()
    final_text = model.finish()
    synchronize_cuda()
    finish_time = time.perf_counter() - finish_started
    return {
        "trace": trace,
        "live": previous_text or "",
        "final": final_text,
        "step_s": step_times,
        "finish_s": finish_time,
    }

out = stream(model, audio, verbose=True)

## 6. The final transcript

Now compare what the model said live, what it said after `finish()`, and the reference.
`finish_s` is how long the final pass took on this machine. CUDA timing waits for queued GPU work to finish.
It varies with hardware, clip length and decoding settings.

In [ ]:
print("LIVE (last) :", out["live"])
print("FINAL       :", out["final"])
print("REFERENCE   :", reference)
print(f"final pass took {out['finish_s'] * 1000:.0f} ms")

## 7. Compare lookahead settings

`lookahead` is fixed when the model is created, so we load the model once per setting
(0, 1, 3, 6, 13) and stream the same clip. For each setting we record:

* **encoder audio wait**: `(lookahead + 1) x 80 ms`, before compute and decoding,
* **first text at**: the audio time at which the first words showed up,
* **last change at**: the audio time of the last change of the live text,
* **updates / rewrites**: how many times the live text changed, and how many of those were *rewrites* (an earlier part of
  the text changed instead of just growing at the end),
* **live text** vs **final text**.

Expect small lookahead to show words sooner, with a few more mistakes or rewrites, and large lookahead to be later but steadier.
We free the GPU memory between runs.

In [ ]:
import gc

def rewrites(trace):
    """Count caption updates that change earlier text."""
    rewrite_count = 0
    for previous, current in zip(trace, trace[1:]):
        previous_text = previous[1].rstrip(" .,?!")
        current_text = current[1]
        if not current_text.startswith(previous_text):
            rewrite_count += 1
    return rewrite_count

del model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

results = {}
for lookahead in [0, 1, 3, 6, 13]:
    current_model = CohereStreaming(
        lookahead=lookahead,
        device=DEVICE,
        dtype=DTYPE,
        final_reencode_max_s=FINAL_REENCODE_MAX_S,
    )
    output = stream(current_model, audio)
    first_text_time = float("nan")
    for audio_time, text in output["trace"]:
        if text.strip():
            first_text_time = audio_time
            break
    output.update(
        nominal_ms=(lookahead + 1) * 80,
        first_text_s=first_text_time,
        last_change_s=output["trace"][-1][0],
        updates=len(output["trace"]),
        rewrites=rewrites(output["trace"]),
    )
    results[lookahead] = output
    del current_model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print(f"{'R':>3} {'audio wait':>10} {'first text':>11} {'last change':>12} {'updates':>8} {'rewrites':>9}")
for lookahead, result in results.items():
    print(
        f"{lookahead:>3} {result['nominal_ms']:>8} ms {result['first_text_s']:>9.2f} s "
        f"{result['last_change_s']:>10.2f} s {result['updates']:>8} {result['rewrites']:>9}"
    )

In [ ]:
print("REFERENCE:", reference, "\n")
for lookahead, result in results.items():
    print(f"R={lookahead:<2} live : {result['live']}")
    print(f"     final: {result['final']}")

For this short clip, the final pass uses the same endpoint lookahead for every setting, so the **final** lines
should match (small numerical differences may occur across hardware). This applies only while the buffered clip
fits `final_reencode_max_s`, which is 30 s here. Larger live lookahead can make the **live** text steadier.

Another useful number is the compute cost per step. For sustained live use, compute must keep up with incoming audio
on average (real-time factor below 1). Individual decoding steps can take longer than 80 ms, so a real application
needs buffering. Check the average on your hardware:

In [ ]:
for lookahead, result in results.items():
    average_ms = 1000 * np.mean(result["step_s"])
    real_time_factor = average_ms / 80
    print(f"R={lookahead:<2} average {average_ms:6.1f} ms per 80 ms chunk -> real-time factor {real_time_factor:.2f}")

## 8. (Optional) A tiny WER check

**WER** (word error rate) is the share of words that are wrong: `(substitutions + deletions + insertions) / words in the
reference`. Lower is better. We stream a few short clips at lookahead 0 and 13 and compare the last *live* text with
the *final* text. We lowercase and remove punctuation first so that "Hello," and "hello" count as equal.
This is a smoke test on a handful of clips, not a benchmark. The full evaluation code is in the
[upstream benchmark](https://github.com/Deep-unlearning/cohere-transcribe-streaming/tree/1e1b41ca259b85d0e2b257b6a794d1b9c1e1ef2d/benchmark).

In [ ]:
import re
import jiwer

def normalize_text(text):
    text = text.lower().replace("-", " ")
    words = re.sub(r"[^a-z' ]", "", text).split()
    return " ".join(words)

clips = []
for row in rows:
    clip_audio = decode(row)
    if len(clip_audio) / SR < FINAL_REENCODE_MAX_S:
        clips.append((clip_audio, row["text"]))
clips = clips[:4]
print(len(clips), "clips,", f"{sum(len(clip_audio) for clip_audio, _ in clips) / SR:.0f} s of audio in total")

wer_rows = {}
for lookahead in [0, 13]:
    current_model = CohereStreaming(
        lookahead=lookahead,
        device=DEVICE,
        dtype=DTYPE,
        final_reencode_max_s=FINAL_REENCODE_MAX_S,
    )
    outputs = [stream(current_model, clip_audio) for clip_audio, _ in clips]
    references = [normalize_text(text) for _, text in clips]
    live_predictions = [normalize_text(output["live"]) for output in outputs]
    final_predictions = [normalize_text(output["final"]) for output in outputs]
    wer_rows[lookahead] = {
        "live": 100 * jiwer.wer(references, live_predictions),
        "final": 100 * jiwer.wer(references, final_predictions),
    }
    del current_model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print(f"{'R':>3} {'live WER %':>11} {'final WER %':>12}")
for lookahead, scores in wer_rows.items():
    print(f"{lookahead:>3} {scores['live']:>11.1f} {scores['final']:>12.1f}")

## Where to go next

For your own audio, load a file with `soundfile`, average its channels if needed, and resample it to 16 kHz with
`librosa`. Then pass 1280-sample float32 chunks to `model.step()`, as above.

[Distill_Streaming_Student.ipynb](https://github.com/Deep-unlearning/smol-audio/blob/master/cohere_transcribe_streaming/Distill_Streaming_Student.ipynb)
walks through teacher labelling and student training.
